## Set up the Phoenix connection and establish a test dataset
### Connects to Phoenix

In [6]:
import os
os.environ.pop("ALL_PROXY", None)
os.environ.pop("all_proxy", None)
from dotenv import load_dotenv
load_dotenv()
os.environ["PHOENIX_CLIENT_HEADERS"] = f"api_key={os.getenv('PHOENIX_API_KEY')}"
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com/s/linkedInArticleGen"

from phoenix.otel import register
from opentelemetry.instrumentation.anthropic import AnthropicInstrumentor
register(project_name="linkedInArticleGen")
AnthropicInstrumentor().instrument()

from phoenix.client import Client
client = Client()
print("Connected to Phoenix")

/home/mouse/artGenT430/venv/lib/python3.12/site-packages/phoenix/otel/otel.py:433: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")
Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: linkedInArticleGen
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/linkedInArticleGen/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'api_key': '****', 'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Connected to Phoenix


### Initiate new dataset (only needs running once when starting new dataset)
Read the transcript from Class 4 from a local file, then creates a dataset in Phoenix with that transcript stored as a re-usable test set. 

Later experiments will feed this test case for later tests for objective comparison of outputs.

*One time*

In [ ]:
## Read the dataset
with open("Class_4_test_sample.txt", "r") as f:
    raw_material = f.read()

## Creates a dataset with the given name and stored experiment
dataset = client.datasets.create_dataset(
    name="initial_test_from_Class_4",
    dataset_description="Lecture transcripts and articles from class 4 for testing initial batch of prompts",
    examples=[
        {
            "input": {"raw_material": raw_material},
            "output": {},
            "metadata": {"source": "class 4 transcript"}
        }
    ]
)
print(f"Created dataset: {dataset.id}")

### Run old dataset

In [ ]:
# Load existing dataset (already created in Phoenix)
dataset = client.datasets.get_dataset(dataset="initial_test_from_Class_4")
print(f"Loaded dataset: {dataset.id}")

# Run the experiment for angleProposalPrompt1.md
## Run the experiment and store the output

In [ ]:
# Load the Phoenix experiment runner
from phoenix.client.experiments import run_experiment
from datetime import datetime
import anthropic

# Helper to read prompt files from the prompts/ folder
def read_prompt(filename):
    with open(f"prompts/{filename}", "r") as f:
        return f.read()

# Load current angle proposal prompt (re-reads from file each time so edits are picked up)
angle_prompt = read_prompt("angleProposalPrompt1.md")

# Define the task — Phoenix runs this against each example in the dataset
def angle_proposal_task(input):
    api_client = anthropic.Anthropic()
    message = api_client.messages.create(
        model="claude-opus-4-5-20251101",
        max_tokens=16000,
        system=angle_prompt,
        messages=[{"role": "user", "content": input["raw_material"]}],
    )
    return message.content[0].text

# Auto-generate unique experiment name from timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"angle-proposal-{timestamp}"

# Run the experiment
latest_experiment = run_experiment(
    dataset=dataset,
    task=angle_proposal_task,
    experiment_name=experiment_name,
    experiment_description="Auto-run",
    timeout=120
)

print(f"Experiment: {experiment_name}")

## Export the experiment in a legible format

### Export the prompt sys message

In [ ]:
# See what was sent to the API as the system prompt
print("=" * 60)
print("SYSTEM PROMPT:")
print("=" * 60)
print(angle_prompt)

### For exporting the user message

In [ ]:
# See the input that was sent as the user message
print("=" * 60)
print("USER MESSAGE (first 500 chars):")
print("=" * 60)
print(raw_material[:500])

### For exporting a single experiment output

In [ ]:
# Pull the latest experiment results in a readable format
for run in latest_experiment["task_runs"]:
    print("=" * 60)
    print("OUTPUT:")
    print("=" * 60)
    print(run["output"])
    print("\n\n")

## Run A/B test on angleProposalPromptV2 with new dataset

### Load new dataset from Economics Explained (run once)
https://www.youtube.com/watch?v=1zlD0K09IKE

In [ ]:
# Read the YouTube transcript
with open("YouTubeEconTranscript.txt", "r") as f:
    raw_material_yt = f.read()

# Create dataset in Phoenix
yt_dataset = client.datasets.create_dataset(
    name="youtube_econ_transcript",
    dataset_description="YouTube AI economics transcript — external material for A/B testing angle proposal prompt with vs without subfield specialization",
    examples=[
        {
            "input": {"raw_material": raw_material_yt},
            "output": {},
            "metadata": {"source": "YouTube AI economics transcript"}
        }
    ]
)
print(f"Created dataset: {yt_dataset.id}")

### Load dataset

In [7]:
yt_dataset = client.datasets.get_dataset(dataset="youtube_econ_transcript")
print(f"Loaded dataset: {yt_dataset.id}")

Loaded dataset: RGF0YXNldDo1


### Run experiment — V1 (control)

In [ ]:
import anthropic
from datetime import datetime
from phoenix.client.experiments import run_experiment

def read_prompt(filename):
    with open(f"prompts/{filename}", "r") as f:
        return f.read()

angle_prompt_v1 = read_prompt("angleProposalPrompt1.md")

def angle_proposal_v1_task(input):
    api_client = anthropic.Anthropic()
    message = api_client.messages.create(
        model="claude-opus-4-5-20251101",
        max_tokens=16000,
        system=angle_prompt_v1,
        messages=[{"role": "user", "content": input["raw_material"]}],
    )
    return message.content[0].text

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"angle-proposal-CONTROL-{timestamp}"
run_description = "V1 — original angle proposal prompt, no subfield specialization"

v1_experiment = run_experiment(
    dataset=yt_dataset,
    task=angle_proposal_v1_task,
    experiment_name=experiment_name,
    experiment_description=run_description,
    timeout=120
)

print(f"\nExperiment: {experiment_name}")
print(f"Description: {run_description}")

#### Print V1 (control) output

In [ ]:
import re

for i, run in enumerate(v1_experiment["task_runs"]):
    output = run["output"]
    # Extract just the Final Pitches section
    match = re.search(r"##\s*Final\s*Pitches", output, re.IGNORECASE)
    pitches = output[match.end():].strip() if match else output
    
    print("=" * 70)
    print(f"V1 CONTROL — RUN {i+1}")
    print("=" * 70)
    print(pitches)
    print("\n" + "#" * 70 + "\n")

## Run experiment — V2 (experiment)

In [ ]:
angle_prompt_v2 = read_prompt("angleProposalPromptV2.md")

def angle_proposal_v2_task(input):
    api_client = anthropic.Anthropic()
    message = api_client.messages.create(
        model="claude-opus-4-5-20251101",
        max_tokens=16000,
        system=angle_prompt_v2,
        messages=[{"role": "user", "content": input["raw_material"]}],
    )
    return message.content[0].text

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"angle-proposal-SUBFIELD-{timestamp}"
run_description = "V2 — angle proposal with Angle Focus section and subfield alignment step"

v2_experiment = run_experiment(
    dataset=yt_dataset,
    task=angle_proposal_v2_task,
    experiment_name=experiment_name,
    experiment_description=run_description,
    timeout=120
)

print(f"\nExperiment: {experiment_name}")
print(f"Description: {run_description}")

## Print V2 (experiment) output

In [ ]:
for i, run in enumerate(v2_experiment["task_runs"]):
    print("=" * 70)
    print(f"V2 SUBFIELD — RUN {i+1} (FULL OUTPUT)")
    print("=" * 70)
    print(run["output"])
    print("\n" + "#" * 70 + "\n")

## Run batch experiment on V2 (experiment) output for angle originality assessment

In [13]:
import anthropic
import re
import time
import os
from datetime import datetime
from openai import OpenAI

def read_prompt(filename):
    with open(f"prompts/{filename}", "r") as f:
        return f.read()

print("Loading prompts...")
angle_prompt_v2 = read_prompt("angleProposalPromptV2.md")
print(f"  Loaded angleProposalPromptV2.md ({len(angle_prompt_v2)} chars)")

JUDGE_SYSTEM = read_prompt("anglePromptOriginalityTest.md")
print(f"  Loaded anglePromptOriginalityTest.md ({len(JUDGE_SYSTEM)} chars)")

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("OpenAI client initialized.")

# --- Extract focus area from V2 prompt ---
focus_match = re.search(r"# Angle Focus\n(.*?)(?=\n# Process)", angle_prompt_v2, re.DOTALL)
focus_area = focus_match.group(1).strip() if focus_match else "See prompt"
print(f"Focus area extracted ({len(focus_area)} chars)")

raw_material = yt_dataset.examples[0]["input"]["raw_material"]
print(f"Raw material loaded ({len(raw_material)} chars)")

# --- Run the angle proposal 3 times ---
print(f"\n{'='*60}")
print("  PHASE 1: GENERATING PITCHES (3 Opus calls)")
print(f"{'='*60}")

runs = []
for batch in range(1, 4):
    print(f"\n  Batch {batch}/3 — Calling Opus...")
    try:
        api_client = anthropic.Anthropic()
        message = api_client.messages.create(
            model="claude-opus-4-5-20251101",
            max_tokens=16000,
            system=angle_prompt_v2,
            messages=[{"role": "user", "content": raw_material}],
        )
        output = message.content[0].text
        runs.append(output)
        print(f"  Batch {batch}/3 — Complete. Output: {len(output)} chars")
    except Exception as e:
        print(f"  ERROR in batch {batch}: {e}")
        runs.append(None)

    if batch < 3:
        print(f"  Waiting 10s before next batch...")
        time.sleep(10)

# Check for failures
failed = [i+1 for i, r in enumerate(runs) if r is None]
if failed:
    print(f"\n  WARNING: Batches {failed} failed. Skipping their evaluation.")

print(f"\n{'='*60}")
print("  PHASE 2: EVALUATING PITCHES (GPT-4o judge)")
print(f"{'='*60}")

# --- Run judge on each batch ---
all_evaluations = []
for batch_num, full_output in enumerate(runs, 1):
    if full_output is None:
        print(f"\n  Batch {batch_num}/3 — Skipped (generation failed)")
        continue

    print(f"\n  Batch {batch_num}/3 — Calling GPT-4o judge...")
    try:
        judge_user_message = f"""## Raw Material
{raw_material}

## Focus Area
{focus_area}

## Model Output
{full_output}"""

        response = openai_client.chat.completions.create(
            model="gpt-4o",
            max_tokens=4000,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM},
                {"role": "user", "content": judge_user_message},
            ],
        )
        evaluation = response.choices[0].message.content
        all_evaluations.append((batch_num, evaluation))
        print(f"  Batch {batch_num}/3 — Evaluation complete. Response: {len(evaluation)} chars")
    except Exception as e:
        print(f"  ERROR evaluating batch {batch_num}: {e}")

    if batch_num < 3:
        time.sleep(5)

print(f"\n{'='*60}")
print(f"  DONE — {len(all_evaluations)} of 3 batches evaluated successfully.")
print(f"  Run the print cell to see results.")
print(f"{'='*60}")

Loading prompts...
  Loaded angleProposalPromptV2.md (17428 chars)
  Loaded anglePromptOriginalityTest.md (6412 chars)
OpenAI client initialized.
Focus area extracted (1572 chars)
Raw material loaded (19922 chars)

  PHASE 1: GENERATING PITCHES (3 Opus calls)

  Batch 1/3 — Calling Opus...
  Batch 1/3 — Complete. Output: 14761 chars
  Waiting 10s before next batch...

  Batch 2/3 — Calling Opus...
  Batch 2/3 — Complete. Output: 18028 chars
  Waiting 10s before next batch...

  Batch 3/3 — Calling Opus...
  Batch 3/3 — Complete. Output: 15154 chars

  PHASE 2: EVALUATING PITCHES (GPT-4o judge)

  Batch 1/3 — Calling GPT-4o judge...
  Batch 1/3 — Evaluation complete. Response: 4547 chars

  Batch 2/3 — Calling GPT-4o judge...
  Batch 2/3 — Evaluation complete. Response: 4385 chars

  Batch 3/3 — Calling GPT-4o judge...
  Batch 3/3 — Evaluation complete. Response: 4719 chars

  DONE — 3 of 3 batches evaluated successfully.
  Run the print cell to see results.


## Print concatenated results

In [14]:
print("\n" + "=" * 70)
print("  ORIGINALITY EVALUATION — ALL BATCHES")
print("=" * 70)

for batch_num, evaluation in all_evaluations:
    print(f"\n{'#'*70}")
    print(f"  BATCH {batch_num}")
    print(f"{'#'*70}\n")
    print(evaluation)

print("\n" + "=" * 70)
print("  EVALUATION COMPLETE")
print("=" * 70)


  ORIGINALITY EVALUATION — ALL BATCHES

######################################################################
  BATCH 1
######################################################################

Pitch [1]: The instruments can't see the problem (Analytical / data breakdown): GDP, unemployment figures, and workforce planning tools explain less than 5% of the variation in actual AI exposure across states. MIT had to build an entirely new measurement approach — skill-weighted wage exposure — just to see the 11.7% of labor market value that existing instruments are blind to. This is an organizational epistemology problem as much as an economic one: the tools we use to know what we know about our workforce were designed for a world where jobs were stable containers for stable tasks.

**Bridge:** The model connected the limitations of traditional economic measures to the need for new organizational frameworks to understand AI's impact on tasks rather than jobs.

**Originality score:** 4 — Non-

# Run the experiment up to supervisorPrompt1.5
## Run the pipeline up to supervisorPrompt

In [ ]:
import time
import re
from datetime import datetime
from phoenix.client.experiments import run_experiment
import anthropic

def read_prompt(filename):
    with open(f"prompts/{filename}", "r") as f:
        return f.read()

def extract_final_pitches(output):
    """Extract only the Final Pitches section, stripping all reasoning."""
    marker = "## Final Pitches"
    idx = output.find(marker)
    if idx != -1:
        return output[idx + len(marker):].strip()
    match = re.search(r"##\s*Final\s*Pitches", output, re.IGNORECASE)
    if match:
        return output[match.end():].strip()
    return output.strip()

def full_angle_pipeline(input):
    api_client = anthropic.Anthropic()
    
    # Load prompts
    angle_prompt = read_prompt("angleProposalPrompt1.md")
    supervisor_prompt = read_prompt("supervisorPrompt1.5.md")
    
    # Run angle proposal 3 times, extract only final pitches
    extracted_pitches = []
    for i in range(1, 4):
        print(f"  → Angle proposal {i}/3...")
        if i > 1:
            time.sleep(10)
        message = api_client.messages.create(
            model="claude-opus-4-5-20251101",
            max_tokens=16000,
            system=angle_prompt,
            messages=[{"role": "user", "content": input["raw_material"]}],
        )
        pitches = extract_final_pitches(message.content[0].text)
        extracted_pitches.append(pitches)
        print(f"  ✓ Batch {i} complete")
    
    # Combine extracted pitches into labeled block
    combined = ""
    for i, pitches in enumerate(extracted_pitches, 1):
        combined += f"## Batch {i}\n\n{pitches}\n\n"
    
    # Run supervisor on combined pitches
    print("  → Running supervisor...")
    supervisor_message = api_client.messages.create(
        model="claude-opus-4-5-20251101",
        max_tokens=16000,
        system=supervisor_prompt,
        messages=[{"role": "user", "content": combined}],
    )
    print("  ✓ Pipeline complete")
    
    return {
        "supervisor_input": combined,
        "supervisor_output": supervisor_message.content[0].text
    }

# ---- CHANGE THIS EACH RUN WHEN NEW DATASET IS EMPLOYED----
run_description = "v1.5 - modded cohesion reviewer 04.06.26"
# --------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"full-pipeline-upToSupervisor-{timestamp}"

latest_experiment = run_experiment(
    dataset=dataset,
    task=full_angle_pipeline,
    experiment_name=experiment_name,
    experiment_description=run_description,
    timeout=1000
)

print(f"Experiment: {experiment_name}")
print(f"Description: {run_description}")

### Print results for the run above

In [ ]:
for run in latest_experiment["task_runs"]:
    output = run["output"]
    
    # print("=" * 60)
    # print("WHAT THE SUPERVISOR RECEIVED:")
    # print("=" * 60)
    # print(output["supervisor_input"])
    
    # print("\n\n")
    
    print("=" * 60)
    print("WHAT THE SUPERVISOR CHOSE:")
    print("=" * 60)
    print(output["supervisor_output"])

In [ ]:
import sys

In [ ]:
print(sys.executable)

# Run the experiment for articleGeneratePrompt2.md

## Initializing dataset
(run once)

### Load the dataset with samples pitches appended

In [ ]:
# Dataset: raw material + all 3 supervisor-selected pitches
with open("Class_4_test_sample.txt", "r") as f:
    raw_material = f.read()

pitches = [
    {
        "title": "Make it sound dumber on purpose",
        "style": "Contrarian",
        "pitch": """Apple had the technology to make Siri sound human in 2013 but deliberately made it robotic. User satisfaction rose because expectations dropped. This isn't a quirk — it's a design principle most AI products ignore. The Bing harassment disaster, the catbot spectrum from "as an AI language model" to "meow meow meow" — every point on that range is technically correct, but only one manages expectations for *your* use case. The argument: the AI that sounds slightly mechanical, gives one-sentence answers, and stops short of what's possible will outperform the one that impresses on first contact."""
    },
    {
        "title": "Your bot forces the org chart soul-search",
        "style": "Observation",
        "pitch": """Most companies have never articulated their organizational voice, logic, or relationship posture — it emerged by accident across years of output. Building a chatbot forces the question. The LUX document isn't a branding exercise; it's archaeology. You're discovering what your company actually believes by trying to make a machine embody it. Industrial firms, nonprofits, anyone who's "just doing the work" — they all hit this wall when the bot needs to talk."""
    },
    {
        "title": "Stop iterating on the chat box",
        "style": "Philosopher",
        "pitch": """We're in horseless carriage territory — bolting AI onto document-writing paradigms because that's what we know. The blank text input isn't intimidating because it lacks examples; it's the wrong interface category entirely. The material shows experiments in coordinate-plane text editors, infinite canvas generators, and comic book makers. The argument: the practitioners who break through aren't optimizing chat UX — they're inventing interaction paradigms that don't exist yet."""
    }
]

article_dataset = client.datasets.create_dataset(
    name="article_gen_raw_plus_pitchTest",
    dataset_description="Supervisor-selected pitches plus raw material from Class 4 for testing article generator prompt",
    examples=[
        {
            "input": {
                "raw_material": raw_material,
                "pitch": p["pitch"],
                "title": p["title"],
                "style": p["style"]
            },
            "output": {},
            "metadata": {"title": p["title"], "style": p["style"]}
        }
        for p in pitches
    ]
)
print(f"Created dataset: {article_dataset.id}")

### Deleting dataset for modifications

In [ ]:
dataset_name = "article_gen_raw_plus_pitchTest"

try:
    # 1. Fetch the dataset to get its ID (this part of the SDK works)
    ds = client.datasets.get_dataset(dataset=dataset_name)
    
    # 2. Use the internal client to send a raw DELETE request
    # This automatically uses the headers and API keys you've already set up
    response = client._client.delete(f"v1/datasets/{ds.id}")
    
    if response.status_code in [200, 204]:
        print(f"Successfully deleted: {dataset_name}")
    else:
        print(f"Failed: {response.status_code} - {response.text}")
except Exception as e:
    print(f"Dataset not found or already deleted: {e}")

### Load dataset (Run every time)

In [ ]:
article_dataset = client.datasets.get_dataset(dataset="article_gen_raw_plus_pitchTest")
print(f"Loaded dataset: {article_dataset.id}")

## Run the experiment and save the output

In [ ]:
import os
import anthropic
from datetime import datetime
from phoenix.client.experiments import run_experiment


def read_prompt(filename):
    with open(f"prompts/{filename}", "r") as f:
        return f.read()

article_prompt = read_prompt("articleGeneratePrompt2.md")

def run_article_generator(example):
    api_client = anthropic.Anthropic()
    
    pitch_block = f"[{example['input']['title']}] | Style: {example['input']['style']}\n{example['input']['pitch']}"
    user_content = f"## Selected Pitch\n{pitch_block}\n\n## Raw Material\n{example['input']['raw_material']}"
    
    print(f"  → Generating articles for: {example['input']['title']}...")
    message = api_client.messages.create(
        model="claude-opus-4-5-20251101",
        max_tokens=16000,
        system=article_prompt,
        messages=[{"role": "user", "content": user_content}],
    )
    print(f"  ✓ Done")
    
    return {"article_versions": message.content[0].text}

# Setup Experiment Metadata
run_description = "v1 - initial article generator test"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"article-generator-{timestamp}"

# Execute experiment
article_experiment = run_experiment(
    dataset=article_dataset,
    task=run_article_generator,
    experiment_name=experiment_name,
    experiment_description=run_description,
    timeout=1000
)

print(f"\nExperiment: {experiment_name}")
print(f"Description: {run_description}")


### Print run output

In [ ]:
import re

examples = article_dataset.examples
example_lookup = {ex["id"]: ex for ex in examples}

# Bird's eye view
print("EXPERIMENT SUMMARY")
print("=" * 70)
for i, run in enumerate(article_experiment["task_runs"]):
    example = example_lookup.get(run["dataset_example_id"])
    title = example["input"]["title"] if example else f"Run {i+1}"
    style = example["input"]["style"] if example else ""
    output = run["output"]["article_versions"]
    
    print(f"\nRUN {i+1}: {title} ({style})")
    approaches = re.findall(r'\*\*Delivery approach:\*\*\s*(.+)', output)
    for j, approach in enumerate(approaches, 1):
        print(f"  Version {j}: {approach}")

print("\n" + "=" * 70 + "\n")

# Full outputs
for i, run in enumerate(article_experiment["task_runs"]):
    example = example_lookup.get(run["dataset_example_id"])
    title = example["input"]["title"] if example else f"Run {i+1}"
    style = example["input"]["style"] if example else ""
    output = run["output"]["article_versions"]
    
    print("=" * 70)
    print(f"RUN {i+1}: {title} ({style})")
    print("=" * 70)
    print(output)
    print("\n" + "#" * 70 + "\n")

### Check system prompt